# MovieLens-100K Novelty Experiments (PopDebias + ColdBridge)

This notebook runs the novelty pipeline described in `kaggle/novelty` on MovieLens-100K using GPU-friendly BGE-M3 embeddings.

It will generate:
- `results/movielens_100k/full_results.csv`
- `results/movielens_100k/best_model.txt`
- `results/figures/*.png`

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/faroukq1/LLM-Sequential-Recommendation.git"
REPO_DIR = Path("/kaggle/working/LLM-Sequential-Recommendation")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "remote", "set-url", "origin", REPO_URL], check=True)

current_branch = subprocess.check_output(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"],
    text=True,
).strip()

status_output = subprocess.check_output(["git", "status", "--porcelain"], text=True).strip()
if status_output:
    print("Local changes detected, skipping git pull to preserve edits.")
else:
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", current_branch], check=True)

head_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("Repo:", REPO_DIR)
print("Branch:", current_branch)
print("Commit:", head_sha)

In [ ]:
import importlib.util
import subprocess
import sys

def pip_install(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

pip_install("--upgrade", "pip")
pip_install("pandas", "numpy", "scipy", "scikit-learn", "matplotlib", "tqdm", "multiprocess", "numba")

if importlib.util.find_spec("tensorflow") is None:
    pip_install("tensorflow==2.13.0")

# The experiment runner can encode BGE-M3 either via FlagEmbedding or a
# robust transformers+torch fallback. We avoid forcing FlagEmbedding here
# because some Kaggle images have version conflicts with transformers.
if importlib.util.find_spec("transformers") is None:
    pip_install("transformers")

if importlib.util.find_spec("torch") is None:
    pip_install("torch")

if importlib.util.find_spec("sentencepiece") is None:
    pip_install("sentencepiece")

print("Dependencies installed.")

In [ ]:
from pathlib import Path
import urllib.request
import zipfile

import numpy as np
import pandas as pd

REPO_DIR = Path("/kaggle/working/LLM-Sequential-Recommendation")
WORK_DIR = Path("/kaggle/working/llmseqrec_novelty_ml100k")
SCRIPT_PATH = REPO_DIR / "kaggle" / "run_novelty_ml100k.py"
CACHE_PATH = WORK_DIR / "embeddings" / "movielens_100k" / "item_embeddings_bge_m3.npy"
DATA_DIR = WORK_DIR / "data"

script_src = SCRIPT_PATH.read_text(encoding="utf-8")
supports_env_fallback = "LLMSEQREC_DISABLE_FLAGEMBEDDING" in script_src

if supports_env_fallback:
    print("Runner already supports transformers fallback. Bootstrap cache cell not needed.")
else:
    print("Legacy runner detected. Building BGE cache to bypass FlagEmbedding import.")

    def ensure_ml100k_files(data_dir: Path) -> tuple[Path, Path]:
        udata = data_dir / "ml-100k" / "u.data"
        uitem = data_dir / "ml-100k" / "u.item"
        if udata.exists() and uitem.exists():
            return udata, uitem

        data_dir.mkdir(parents=True, exist_ok=True)
        archive = data_dir / "ml-100k.zip"
        if not archive.exists():
            urllib.request.urlretrieve(
                "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
                archive,
            )
        with zipfile.ZipFile(archive, "r") as zf:
            zf.extractall(data_dir)
        return udata, uitem

    def sessionized_item_ids(udata_path: Path, gap_minutes: int = 30, min_session_len: int = 2) -> np.ndarray:
        interactions = pd.read_csv(
            udata_path,
            sep="\t",
            names=["UserId", "ItemId", "Rating", "Timestamp"],
            engine="python",
        )

        df = interactions.copy()
        df["Timestamp"] = df["Timestamp"].astype(int)
        df["ItemId"] = df["ItemId"].astype(int)
        df = df.sort_values(["UserId", "Timestamp", "ItemId"]).reset_index(drop=True)

        gap_seconds = gap_minutes * 60
        user_changed = df["UserId"].ne(df["UserId"].shift(1))
        ts_gap = df["Timestamp"].diff().fillna(0)
        new_session = user_changed | (ts_gap > gap_seconds)
        df["SessionId"] = new_session.cumsum().astype(int)

        session_sizes = df.groupby("SessionId").size()
        valid_sessions = session_sizes[session_sizes >= min_session_len].index
        df = df[df["SessionId"].isin(valid_sessions)].copy()

        return np.array(sorted(df["ItemId"].unique().tolist()), dtype=int)

    def build_item_texts(uitem_path: Path, target_item_ids: np.ndarray) -> list[str]:
        genre_cols = [
            "unknown", "action", "adventure", "animation", "childrens", "comedy",
            "crime", "documentary", "drama", "fantasy", "film_noir", "horror",
            "musical", "mystery", "romance", "sci_fi", "thriller", "war", "western",
        ]
        cols = [
            "ItemId", "title", "release_date", "video_release_date", "imdb_url", *genre_cols,
        ]

        item_df = pd.read_csv(
            uitem_path,
            sep="|",
            names=cols,
            encoding="latin-1",
            usecols=list(range(len(cols))),
            engine="python",
        )

        def make_text(row: pd.Series) -> str:
            genres = [g.replace("_", " ") for g in genre_cols if int(row[g]) == 1]
            genre_text = ", ".join(genres) if genres else "unknown"
            return f"{row['title']}. Genres: {genre_text}."

        item_df["item_text"] = item_df.apply(make_text, axis=1)
        text_lookup = item_df.set_index("ItemId")["item_text"].to_dict()
        return [text_lookup.get(int(i), f"Movie item {int(i)}.") for i in target_item_ids]

    def mean_pool(last_hidden_state, attention_mask):
        import torch

        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        masked = last_hidden_state * mask
        summed = masked.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    udata_path, uitem_path = ensure_ml100k_files(DATA_DIR)
    target_item_ids = sessionized_item_ids(udata_path)

    cache_ok = False
    if CACHE_PATH.exists():
        existing = np.load(CACHE_PATH)
        cache_ok = existing.shape[0] == len(target_item_ids)
        if cache_ok:
            print(f"Existing cache matches ({existing.shape}).")
        else:
            print(f"Cache shape mismatch {existing.shape}, regenerating...")

    if not cache_ok:
        import torch
        import torch.nn.functional as F
        from transformers import AutoModel, AutoTokenizer

        texts = build_item_texts(uitem_path, target_item_ids)

        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.float16 if device == "cuda" else torch.float32

        tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
        model = AutoModel.from_pretrained("BAAI/bge-m3", torch_dtype=dtype).to(device)
        model.eval()

        batch_size = 128
        chunks = []
        for start in range(0, len(texts), batch_size):
            cur = texts[start:start + batch_size]
            encoded = tokenizer(
                cur,
                padding=True,
                truncation=True,
                max_length=256,
                return_tensors="pt",
            )
            encoded = {k: v.to(device) for k, v in encoded.items()}

            with torch.no_grad():
                out = model(**encoded)
                pooled = mean_pool(out.last_hidden_state, encoded["attention_mask"] )
                pooled = F.normalize(pooled, p=2, dim=1)

            chunks.append(pooled.detach().cpu().numpy().astype(np.float32))

        embeddings = np.vstack(chunks).astype(np.float32)
        CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
        np.save(CACHE_PATH, embeddings)
        print(f"Saved bootstrap cache: {CACHE_PATH} with shape {embeddings.shape}")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = "/kaggle/working/LLM-Sequential-Recommendation"
WORK_DIR = "/kaggle/working/llmseqrec_novelty_ml100k"
SCRIPT_PATH = f"{REPO_DIR}/kaggle/run_novelty_ml100k.py"

script_src = Path(SCRIPT_PATH).read_text(encoding="utf-8")
legacy_runner = "LLMSEQREC_DISABLE_FLAGEMBEDDING" not in script_src
legacy_cache = Path(WORK_DIR) / "embeddings" / "movielens_100k" / "item_embeddings_bge_m3.npy"
if legacy_runner and not legacy_cache.exists():
    raise RuntimeError(
        "Legacy novelty runner detected and no cached BGE embeddings found. "
        "Run Cell 4 first (the compatibility bootstrap cell), then rerun this cell."
    )

# Toggle this if you want the optional BERT4Rec backbone comparison.
INCLUDE_BERT_BACKBONE = False

cmd = [
    sys.executable,
    SCRIPT_PATH,
    "--work-dir", WORK_DIR,
    "--num-epochs", "6",
    "--cores", "2",
    "--top-k", "20",
    "--candidate-k", "300",
    "--alpha-values", "0.1,0.3,0.5,0.7",
    "--tau-values", "2,3,5,10,15",
    "--decay-values", "0.5,0.7,0.8,0.9,1.0",
    "--bge-batch-size", "256",
]

if INCLUDE_BERT_BACKBONE:
    cmd.append("--include-bert-backbone")

env = os.environ.copy()
if env.get("PYTHONPATH"):
    env["PYTHONPATH"] = f"{REPO_DIR}:{env['PYTHONPATH']}"
else:
    env["PYTHONPATH"] = REPO_DIR

# Force robust encoder path on Kaggle when FlagEmbedding has version conflicts.
env["LLMSEQREC_DISABLE_FLAGEMBEDDING"] = "1"

print("Running command:")
print(" ".join(cmd))
subprocess.run(cmd, check=True, cwd=REPO_DIR, env=env)
print("Done. Artifacts written to:", WORK_DIR)

In [ ]:
from pathlib import Path
import pandas as pd

work_dir = Path("/kaggle/working/llmseqrec_novelty_ml100k")
results_csv = work_dir / "results" / "movielens_100k" / "full_results.csv"
best_txt = work_dir / "results" / "movielens_100k" / "best_model.txt"
figures_dir = work_dir / "results" / "figures"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    display(df.head(20))
else:
    print("Results file not found:", results_csv)

if best_txt.exists():
    print("=== best_model.txt ===")
    print(best_txt.read_text(encoding="utf-8"))
else:
    print("Best-model file not found:", best_txt)

if figures_dir.exists():
    print("Figures:")
    for p in sorted(figures_dir.glob("*.png")):
        print("-", p.name)

If Kaggle time is tight, reduce training cost first by setting `--num-epochs 3` and `INCLUDE_BERT_BACKBONE = False`.